# Colab CNN Improvement Experiments

This notebook keeps the same predefined train/eval/test CSV splits as notebook 06, then tests stronger CNN baseline variants. The goal is to improve the CNN baseline scientifically without changing the dataset split or comparing models on different data.

What stays the same as notebook 06:

- same GSE144621 / EP DNABERT2 genomic-order split files;
- same `train`, `eval`/validation, and `test` meaning;
- same seed and sequence field/label field from the benchmark config;
- same rule that thresholds are chosen on validation, not test;
- same standard comparison metrics used later for DNABERT2 and iPro-MP.

What is new in notebook 07:

- longer-cycle tiny CNN checks to see whether notebook 06 was under-trained;
- AdamW and OneCycleLR experiments for more stable optimization;
- best-validation-MCC checkpoint selection and optional early stopping;
- enhanced CNN variants with wider motif filters, batch normalization, dropout, dilated convolutions, and average+max pooling;
- full metric ranking so we choose by validation MCC and only report test metrics after that choice.

The most important rule here is: train on `train`, choose thresholds and candidate models on `eval`/validation, and report final scores on `test`.

If notebook 07 clearly beats notebook 06 on validation MCC and then improves held-out test MCC/AUPRC, the selected 07 configuration becomes the improved CNN baseline candidate. Notebook 06 still remains the original/reference baseline.


## 1. Set Up SeqTrainer In Colab

This clones or updates the working branch, installs the package, and verifies that `seqtrainer` imports before any benchmark cell runs.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "issue-3-cnn-baseline-reproduction"
REPO_DIR = Path("/content/SeqTrainer")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo already exists at {REPO_DIR}; updating {BRANCH}")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print("Python import path includes:", SRC_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[torch]"], check=True)

import seqtrainer
print("SeqTrainer import OK:", seqtrainer.__file__)

## 2. Mount Drive And Prepare The Three Dataset Files

This uses the same files as notebook 06. If Colab cannot see the Drive path, it extracts the same CSVs from the repo zip so the benchmark can still run.

In [ ]:
from pathlib import Path
import shutil
import zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("google.colab is not available. Assuming dataset files are already local or in the repo zip.")

LOCAL_DATA_DIR = Path("data/promoter_classification")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

split_file_names = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

expected_relative_dir = Path("AIxBio") / "Promoter Classification" / "Data"
drive_roots = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/My Drive"),
    Path("/content/drive/Shareddrives"),
]


def has_all_split_files(directory: Path) -> bool:
    return all((directory / file_name).exists() for file_name in split_file_names.values())


def find_drive_data_dir() -> Path | None:
    candidates = [root / expected_relative_dir for root in drive_roots]
    print("Checking expected Drive paths:")
    for candidate in candidates:
        print(f"  - {candidate}")
        if has_all_split_files(candidate):
            return candidate

    search_roots = []
    for root in drive_roots:
        aixbio = root / "AIxBio"
        if aixbio.exists():
            search_roots.append(aixbio)
    for root in search_roots:
        print(f"Searching for train CSV under: {root}")
        for train_path in root.rglob(split_file_names["train"]):
            candidate = train_path.parent
            if has_all_split_files(candidate):
                return candidate
    return None


def copy_from_drive(data_dir: Path) -> bool:
    for split, file_name in split_file_names.items():
        source = data_dir / file_name
        target = LOCAL_DATA_DIR / file_name
        shutil.copy2(source, target)
        print(f"Copied {split}: {target}")
    return True


def extract_from_repo_zip() -> bool:
    zip_path = Path("data/data_DNABERT/promoter_classification_DNABERT.zip")
    if not zip_path.exists():
        return False

    print(f"Drive CSVs were not found locally. Extracting from repo zip: {zip_path}")
    with zipfile.ZipFile(zip_path) as zf:
        members = set(zf.namelist())
        for split, file_name in split_file_names.items():
            if file_name not in members:
                raise FileNotFoundError(f"{file_name} is missing from {zip_path}")
            target = LOCAL_DATA_DIR / file_name
            with zf.open(file_name) as source, target.open("wb") as dest:
                shutil.copyfileobj(source, dest)
            print(f"Extracted {split}: {target}")
    return True


drive_data_dir = find_drive_data_dir()
if drive_data_dir is not None:
    print(f"Using Drive data directory: {drive_data_dir}")
    copy_from_drive(drive_data_dir)
elif all((LOCAL_DATA_DIR / file_name).exists() for file_name in split_file_names.values()):
    print(f"Using existing local CSV files in: {LOCAL_DATA_DIR}")
elif extract_from_repo_zip():
    print(f"Using CSV files extracted into: {LOCAL_DATA_DIR}")
else:
    raise FileNotFoundError(
        "Could not find the promoter CSV files in Drive or in the repo zip. "
        "Expected Drive folder: /content/drive/MyDrive/AIxBio/Promoter Classification/Data"
    )

## 3. Verify Data And Benchmark Contract

This confirms that the same dataset, fields, and splits are used for all CNN variants. It also checks label balance. Class weighting is used only when the training split appears meaningfully imbalanced.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/SeqTrainer")
SRC_DIR = REPO_DIR / "src"
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
from seqtrainer.benchmarks import load_benchmark_config

CONFIG_PATH = Path("config-examples/benchmarks/cnn.toml")
config = load_benchmark_config(CONFIG_PATH)

print("Dataset:", config.dataset.name)
print("Source accession:", config.dataset.source_accession)
print("Split strategy:", config.split.strategy)
print("Primary metric:", config.evaluation.primary_metric)

IMBALANCE_RATIO_THRESHOLD = 1.5
split_balance_rows = []
TRAIN_SPLIT_IMBALANCE_RATIO = None

for split, csv_path in config.dataset.split_files.items():
    frame = pd.read_csv(csv_path)
    label_counts = frame[config.dataset.label_field].value_counts().sort_index().to_dict()
    sequence_lengths = frame[config.dataset.sequence_field].astype(str).str.len()
    majority_count = max(label_counts.values())
    minority_count = min(label_counts.values())
    imbalance_ratio = majority_count / max(minority_count, 1)
    balance_note = "imbalanced" if imbalance_ratio >= IMBALANCE_RATIO_THRESHOLD else "balanced"
    split_balance_rows.append(
        {
            "split": split,
            "rows": len(frame),
            "label_counts": label_counts,
            "imbalance_ratio": imbalance_ratio,
            "balance_note": balance_note,
        }
    )
    if split == "train":
        TRAIN_SPLIT_IMBALANCE_RATIO = imbalance_ratio
    print(
        f"{split}: rows={len(frame)} labels={label_counts} "
        f"imbalance_ratio={imbalance_ratio:.3f} balance={balance_note} "
        f"length_min={sequence_lengths.min()} length_max={sequence_lengths.max()} "
        f"length_mean={sequence_lengths.mean():.1f}"
    )

USE_CLASS_WEIGHTING = TRAIN_SPLIT_IMBALANCE_RATIO >= IMBALANCE_RATIO_THRESHOLD
print("Use class weighting for notebook 07 experiments:", USE_CLASS_WEIGHTING)
display(pd.DataFrame(split_balance_rows))

## 4. Define CNN Improvement Experiments

These experiments are controlled rather than exhaustive. They test changes that are scientifically interpretable:

- more cycles for the original tiny CNN: checks whether the baseline was simply under-trained;
- AdamW, OneCycleLR, and best-validation-MCC checkpointing: aligns training with the benchmark's MCC-centered evaluation;
- enhanced CNN: adds a wider first motif filter, batch normalization, dropout, dilated convolution blocks, and average+max pooling;
- regularized enhanced CNN: lowers learning rate and adds stronger dropout/weight decay to reduce overfitting risk;
- class weighting: enabled automatically only if the training split is meaningfully imbalanced.

Notebook 06 and notebook 07 do not need the same number of cycles because they have different jobs. Notebook 06 preserves the original reference run. Notebook 07 is allowed to test longer training, but it includes longer-cycle tiny-CNN controls so we can tell whether improvement came from more training alone or from the stronger model/training design.

You can increase `cycles` later, but start with this grid so each change has a clear reason.


In [ ]:
import torch
from seqtrainer.torch.cnn_baseline import CnnCsvSplitConfig, run_cnn_csv_splits

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEQUENCE_LENGTH = config.preprocessing.sequence_length or 300
BASE_OUTPUT_DIR = Path("outputs/cnn_improvement_experiments")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Sequence length:", SEQUENCE_LENGTH)

EXPERIMENTS = [
    {
        "name": "tiny_30_cycles",
        "model_variant": "tiny",
        "cycles": 30,
        "batch_size": 32,
        "learning_rate": 1e-3,
        "weight_decay": 0.0,
        "optimizer_name": "adam",
        "scheduler_name": "none",
        "select_best_by_mcc": False,
        "early_stopping_patience": None,
        "dropout": 0.25,
        "class_weighting": USE_CLASS_WEIGHTING,
    },
    {
        "name": "tiny_50_cycles_adamw_best_mcc",
        "model_variant": "tiny",
        "cycles": 50,
        "batch_size": 32,
        "learning_rate": 5e-4,
        "weight_decay": 1e-4,
        "optimizer_name": "adamw",
        "scheduler_name": "one_cycle",
        "select_best_by_mcc": True,
        "early_stopping_patience": 8,
        "dropout": 0.25,
        "class_weighting": USE_CLASS_WEIGHTING,
    },
    {
        "name": "enhanced_30_cycles_best_mcc",
        "model_variant": "enhanced",
        "cycles": 30,
        "batch_size": 32,
        "learning_rate": 5e-4,
        "weight_decay": 1e-4,
        "optimizer_name": "adamw",
        "scheduler_name": "one_cycle",
        "select_best_by_mcc": True,
        "early_stopping_patience": 8,
        "dropout": 0.25,
        "class_weighting": USE_CLASS_WEIGHTING,
    },
    {
        "name": "enhanced_50_cycles_regularized",
        "model_variant": "enhanced",
        "cycles": 50,
        "batch_size": 32,
        "learning_rate": 3e-4,
        "weight_decay": 2e-4,
        "optimizer_name": "adamw",
        "scheduler_name": "one_cycle",
        "select_best_by_mcc": True,
        "early_stopping_patience": 10,
        "dropout": 0.30,
        "class_weighting": USE_CLASS_WEIGHTING,
    },
]

# For a very fast smoke test, set this to 1 or 2. Use None for the full grid.
MAX_EXPERIMENTS = None
experiments_to_run = EXPERIMENTS if MAX_EXPERIMENTS is None else EXPERIMENTS[:MAX_EXPERIMENTS]

pd.DataFrame(experiments_to_run)

## 5. Run The Experiment Grid

Each run writes its own `metrics.csv`, `history.csv`, `manifest.json`, and predictions under `outputs/cnn_improvement_experiments/<experiment_name>/`. Thresholds are chosen on validation MCC and then applied to train/validation/test for reporting.

In [ ]:
all_metrics = []
all_histories = []

for exp in experiments_to_run:
    print()
    print("=== Running", exp["name"], "===")
    output_dir = BASE_OUTPUT_DIR / exp["name"]

    run_config = CnnCsvSplitConfig(
        train_csv=config.dataset.split_files["train"],
        validation_csv=config.dataset.split_files["validation"],
        test_csv=config.dataset.split_files["test"],
        output_dir=output_dir,
        dataset_name=config.dataset.name,
        source_accession=config.dataset.source_accession,
        source_url=config.dataset.source_url,
        sequence_field=config.dataset.sequence_field,
        label_field=config.dataset.label_field,
        sequence_length=SEQUENCE_LENGTH,
        seed=config.experiment.seed,
        batch_size=exp["batch_size"],
        cycles=exp["cycles"],
        learning_rate=exp["learning_rate"],
        weight_decay=exp["weight_decay"],
        optimizer_name=exp["optimizer_name"],
        scheduler_name=exp["scheduler_name"],
        select_best_by_mcc=exp["select_best_by_mcc"],
        early_stopping_patience=exp["early_stopping_patience"],
        model_variant=exp["model_variant"],
        dropout=exp["dropout"],
        class_weighting=exp["class_weighting"],
        device=DEVICE,
    )

    result = run_cnn_csv_splits(run_config)
    metrics_df = pd.read_csv(Path(result.output_dir) / "metrics.csv")
    history_df = pd.read_csv(Path(result.output_dir) / "history.csv")
    metrics_df.insert(0, "experiment", exp["name"])
    history_df.insert(0, "experiment", exp["name"])
    all_metrics.append(metrics_df)
    all_histories.append(history_df)

summary_df = pd.concat(all_metrics, ignore_index=True)
history_df = pd.concat(all_histories, ignore_index=True)
summary_path = BASE_OUTPUT_DIR / "summary_metrics.csv"
history_path = BASE_OUTPUT_DIR / "summary_history.csv"
summary_df.to_csv(summary_path, index=False)
history_df.to_csv(history_path, index=False)
print("Wrote:", summary_path)
print("Wrote:", history_path)

## 6. Compare Full Metrics

Use validation metrics to choose the candidate model. Use test metrics only once for the final estimate. The table includes accuracy, balanced accuracy, precision, recall/sensitivity, F1, MCC, ROC-AUC, AUPRC, specificity, confusion matrix values, and loss.

Decision order:

1. Choose candidates by validation MCC.
2. Use validation AUPRC and balanced accuracy as supporting tie-breakers.
3. After the candidate is chosen, inspect test MCC and test AUPRC for the final claim.
4. Report accuracy, but do not use plain accuracy alone to decide the baseline.


In [ ]:
metric_cols = [
    "experiment",
    "split",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "mcc",
    "sensitivity",
    "specificity",
    "tn",
    "fp",
    "fn",
    "tp",
    "auroc",
    "auprc",
    "loss",
]
existing_cols = [col for col in metric_cols if col in summary_df.columns]

validation_rank = summary_df[summary_df["split"] == "validation"].sort_values(
    ["mcc", "auprc", "balanced_accuracy"], ascending=False
)
test_rank = summary_df[summary_df["split"] == "test"].sort_values(
    ["mcc", "auprc", "balanced_accuracy"], ascending=False
)

print("Validation ranking: choose model here")
display(validation_rank[existing_cols])

print("Test ranking: report after validation choice")
display(test_rank[existing_cols])

## 7. Inspect Training Curves

If training accuracy keeps rising while validation accuracy or validation MCC stalls, the model is overfitting. In that case, prefer the regularized variant or reduce cycles.

In [ ]:
display(history_df.groupby("experiment").tail(5))

try:
    import matplotlib.pyplot as plt

    for exp_name, group in history_df.groupby("experiment"):
        plt.figure(figsize=(7, 4))
        plt.plot(group["cycle"], group["train_loss"], label="train_loss")
        plt.plot(group["cycle"], group["validation_loss"], label="validation_loss")
        plt.title(exp_name)
        plt.xlabel("cycle")
        plt.ylabel("loss")
        plt.legend()
        plt.show()
except Exception as exc:
    print("Plotting skipped:", exc)

## 8. Follow-Up Questions Before Choosing A Better CNN

Use these questions after the grid finishes:

1. Did validation MCC improve, or only raw accuracy?
2. Is the current dataset balanced enough to leave class weighting off?
3. If class weighting was enabled, did it improve MCC/AUPRC without damaging specificity or precision?
4. Did validation AUPRC and balanced accuracy support the MCC improvement?
5. Did test MCC and AUPRC improve together after choosing the candidate on validation?
6. Did recall/sensitivity improve without collapsing specificity?
7. Is the selected threshold scientifically reasonable, or is it compensating for poorly calibrated probabilities?
8. Do `predictions.csv` confusion counts match `metrics.csv` after thresholding?
9. Does the stronger CNN overfit, meaning train loss improves while validation MCC stalls or drops?
10. Does the improved model beat both notebook 06 and the longer-cycle tiny-CNN controls?
11. Are gains stable enough to repeat across more seeds before we compare against DNABERT2?
12. If the improved CNN wins, should we name it `CNN v2 baseline` while keeping notebook 06 as the original baseline reference?


## 9. Optional: Save Improvement Outputs Back To Drive

This preserves the experiment grid outputs after the Colab runtime disconnects.

In [ ]:
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/AIxBio/Promoter Classification/Outputs/cnn_improvement_experiments")

if SAVE_TO_DRIVE:
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    for source in BASE_OUTPUT_DIR.rglob("*"):
        if source.is_file():
            target = DRIVE_OUTPUT_ROOT / source.relative_to(BASE_OUTPUT_DIR)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
    print("Saved outputs to:", DRIVE_OUTPUT_ROOT)